# Building a Sentiment Analysis Classifier for Production

In [ ]:
##making basic imports
import pandas as pd
import kagglehub
import os
import sys
import numpy

In [ ]:
#importing data
path = kagglehub.dataset_download("cosmos98/twitter-and-reddit-sentimental-analysis-dataset")

print(f'Path to data in root: {path}')

!ls $path

In [ ]:
#load data with pandas
df = pd.read_csv(f'{path}/Reddit_Data.csv')

df.head()

In [ ]:
##check the data size
print(f'There are {df.shape[0]} instances and {df.shape[1]} features')

In [ ]:
 #set maximum column width for full view of column content
pd.set_option('display.max_colwidth',None)

df['clean_comment'].iloc[0]

In [ ]:
df.info()

In [ ]:
#check for missing vbalues
df.isnull().sum()

In [ ]:
#check specific rows that have missing values
df[df['clean_comment'].isna()]

In [ ]:
#check for percentage of missing values
total_df = len(df)
missing = df.isna().sum()
portion = (missing/total_df)*100
print(f'Percentage of missing values: {portion}')

In [ ]:
##drop the instances
df.dropna(inplace=True)

In [ ]:
#check instance count
df.shape

In [ ]:
#check for duplicated data
df.duplicated().sum()

In [ ]:
#view the duplicates
df[df.duplicated()]

In [ ]:
#lets drop the duplicates
df.drop_duplicates(inplace=True)

In [ ]:
#check that it worked
df.duplicated().sum()

In [ ]:
##since it is text we check for empty or whitespace
df[(df['clean_comment'].str.strip()=='')]

In [ ]:
#rewmove the instances with whitespaces
df = df[~(df['clean_comment'].str.strip()=='')]

#verify
df[(df['clean_comment'].str.strip() == '')]

In [ ]:
#convert to lowercase
df['clean_comment'] = df['clean_comment'].str.lower()

In [ ]:
df.head()

In [ ]:
##check if there are any empty spaces before or after text
df[df['clean_comment'].apply(lambda x : x.endswith(' ') or x.startswith(' '))]

In [ ]:
##handle the empty spaces
df['clean_comment'] = df['clean_comment'].str.strip()

#verify
df['clean_comment'].apply(lambda x: x.endswith(' ') or x.startswith(' ')).sum()

In [ ]:
##identify urls with regex pattern search
url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
comments_with_url = df[df['clean_comment'].str.contains(url_pattern,regex=True)]

comments_with_url.head()

In [ ]:
##identify comments with newlines
comments_with_newlines = df[df['clean_comment'].str.contains('\n')]

comments_with_newlines.head()

In [ ]:
df.loc[:, 'clean_comment'] = df['clean_comment'].str.replace('\n', '', regex=False)

# Verify that newlines are removed
comments_with_newlines = df[df['clean_comment'].str.contains('\n')]
print(f"Number of comments with newlines after cleaning: {len(comments_with_newlines)}")

In [ ]:
df[df['clean_comment'].str.contains('\n',regex=True)]

EDA

In [ ]:
import seaborn as sns

sns.countplot(data=df,x='category')

In [ ]:
df['category'].value_counts(normalize=True).mul(100).round(2)

In [ ]:
#check the count of words per instance
df['word_count'] = df['clean_comment'].apply(lambda x : len(x.split()))

df.head()

In [ ]:
df['word_count'].describe()

In [ ]:
sns.displot(df['word_count'],kde=True)

In [ ]:
##more detailed plot
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

sns.kdeplot(df[df['category']==1]['word_count'],label='Positive',fill=True)
sns.kdeplot(df[df['category']==0]['word_count'],label='Neutral',fill=True)
sns.kdeplot(df[df['category']==-1]['word_count'],label='Negative',fill=True)

plt.title('Word Count Distribution')
plt.xlabel('Word count')
plt.ylabel('Density')

plt.legend()
plt.show()

In [ ]:
!pip install nltk -q

In [ ]:
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')

stopwords_identified = set(stopwords.words(('english')))

df['num_stop_words'] = df['clean_comment'].apply(lambda x : len([word for word in x.split() if word in stopwords_identified]))

df.head()

In [ ]:
##lets view the stopword count ditribution per category
plt.figure(figsize=(10,6))

sns.kdeplot(df[df['category']==1]['num_stop_words'],label='Positive',fill=True)
sns.kdeplot(df[df['category']==0]['num_stop_words'],label='Neutral',fill=True)
sns.kdeplot(df[df['category']==-1]['num_stop_words'],label='Negative',fill=True)

plt.title('Stopword count by category')
plt.xlabel('num stop word count')
plt.ylabel('desnsity')

plt.legend()
plt.show()

In [ ]:
#frequency distribution of stop words in the clean_comment
from collections import Counter

all_stop_words = [word for comment in df['clean_comment'] for word in comment.split() if word in stopwords_identified]

#count the most common stopwords
most_common_stopwords = Counter(all_stop_words).most_common(25)

#convert the most common stop words to a dataframe
top_25_df = pd.DataFrame(most_common_stopwords,columns=['stop_word','count'])

plt.figure(figsize=(12,8))
sns.barplot(data=top_25_df,x='count',y='stop_word',palette='viridis')
plt.title('top 25 most common stopwords')
plt.xlabel('couint')
plt.ylabel('stopword')
plt.show()


In [ ]:
#num of chars per instances
df['num_chars'] = df['clean_comment'].apply(len)

df.head()

In [ ]:
##character frequerncy

all_text = ' '.join(df['clean_comment'])

char_freq = Counter(all_text)

char_freq_df = pd.DataFrame(char_freq.items(),columns=['char','freq']).sort_values(by='freq',ascending=True)

char_freq_df.head()

In [ ]:
##check for punctuation characters
df['num_punc_chars'] = df['clean_comment'].apply(lambda x : sum([1 for char in x if char in '.!?;"\'()[]{}-']))

df.sample(6)

In [ ]:
##remove all special characters
import re

df['clean_comment'] = df['clean_comment'].apply(lambda x : re.sub(r'[^A-Za-z0-9\s!?.,]','',str(x)))

df.sample(5)

In [ ]:
all_text = ' '.join(df['clean_comment'])

char_freq = Counter(all_text)

char_freq_df = pd.DataFrame(char_freq.items(),columns=['char','freq']).sort_values(by='freq',ascending=True)

char_freq_df.sample(10)

In [ ]:
##remove stopwords but keep soe
stopwords_ = set(stopwords.words('english'))-{'not','no','however','but','yet'}

df['clean_comment'] = df['clean_comment'].apply(lambda x : ' '.join([word for word in x.split() if word not in stopwords_]))

df.sample(5)

In [ ]:
from nltk.stem.wordnet import WordNetLemmatizer

nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

df['clean_comment'] = df['clean_comment'].apply(lambda x : ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))

df.sample(5)

In [ ]:
##performing wordcloud
from wordcloud import WordCloud

def plot_wordcloud(text:pd.DataFrame):
    wordcloud = WordCloud(width=800,height=500,background_color='black').generate(' '.join(text))
    plt.figure(figsize=(12,8))
    plt.imshow(wordcloud,interpolation='bilinear')
    plt.axis('off')
    plt.show()
plot_wordcloud(df['clean_comment'])

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_top_words_by_sentiment(df: pd.DataFrame,
                                N: int = 20,
                                start: int = 0,
                                orientation: str = 'vertical',
                                palette:str='viridis'):
    """
    Plots the top N most frequent words from the 'clean_comment' column,
    starting from a specified offset, with sentiment categories as hues.

    Args:
        df (pd.DataFrame): The input DataFrame containing 'clean_comment' and 'category' columns.
        N (int, optional): The number of top words to plot. Defaults to 20.
        start (int, optional): The starting index for selecting words (e.g., start=0 for top words,
                               start=10 for words from 11th to (10+N)th). Defaults to 0.
        orientation (str, optional): The orientation of the bars ('vertical' or 'horizontal'). Defaults to 'vertical'.
    """

    # Create a list of (word, category) tuples
    word_category_pairs = []
    for index, row in df.iterrows():
        words = row['clean_comment'].split()
        category = row['category']
        word_category_pairs.extend([(word, category) for word in words])

    # Create a DataFrame from these pairs
    words_df_flat = pd.DataFrame(word_category_pairs, columns=['word', 'category'])

    # Get overall word frequencies to find top N words from the specified start index
    overall_word_counts = words_df_flat['word'].value_counts()
    top_n_words_series = overall_word_counts.iloc[start : start + N]
    top_n_words = top_n_words_series.index.tolist()

    # Filter the flat DataFrame to only include selected top words
    filtered_words_df = words_df_flat[words_df_flat['word'].isin(top_n_words)]

    # Group by word and category to get counts
    plot_df = filtered_words_df.groupby(['word', 'category']).size().reset_index(name='count')

    # Ensure all selected top words appear for all categories, even if count is 0, for proper stacking
    all_combinations = pd.MultiIndex.from_product([top_n_words, df['category'].unique()], names=['word', 'category']).to_frame(index=False)
    plot_df = pd.merge(all_combinations, plot_df, on=['word', 'category'], how='left').fillna(0)
    plot_df['count'] = plot_df['count'].astype(int)

    # Order the words by overall frequency for consistent plotting (based on the selected N words)
    word_order = top_n_words_series.index

    # Plotting
    plt.figure(figsize=(15, 8))
    if orientation == 'vertical':
        sns.barplot(
            data=plot_df,
            x='word',
            y='count',
            hue='category',
            order=word_order,
            palette=f'{palette}',
            hue_order=[1, 0, -1]
        )
        plt.xlabel('Word')
        plt.ylabel('Frequency')
        plt.xticks(rotation=45, ha='right')
    elif orientation == 'horizontal':
        sns.barplot(
            data=plot_df,
            x='count',
            y='word',
            hue='category',
            order=word_order[::-1], # Reverse order for horizontal plot to have highest at top
            palette='viridis',
            hue_order=[1, 0, -1]
        )
        plt.xlabel('Frequency')
        plt.ylabel('Word')
        plt.yticks(rotation=0)
    else:
        print("Invalid orientation. Please choose 'vertical' or 'horizontal'.")
        return

    plt.title(f'Top {N} Most Frequent Words (starting from rank {start + 1}) by Sentiment Category')
    plt.legend(title='Sentiment Category', labels=['Positive (1)', 'Neutral (0)', 'Negative (-1)'])
    plt.tight_layout()
    plt.show()

plot_top_words_by_sentiment(df,20,start=0,palette='coolwarm')

## Experiment tracking

In [ ]:
!pip install mlflow -q

In [ ]:
#import mlflow and configure the tracking uri
#this is a dummy experiment
import mlflow

mlflow.set_tracking_uri('http://ec2-3-88-103-118.compute-1.amazonaws.com:5000/')

#start run
with mlflow.start_run():
    mlflow.log_param('param1',15)
    mlflow.log_metric('metric1',90)

In [ ]:
#import necessary libraies for mdoel condig
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfTransformer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,precision_recall_fscore_support

In [ ]:
#initialize the vectorizer for feature engineering
vectorizer = CountVectorizer(max_features=10000)

X = vectorizer.fit_transform(df['clean_comment']).toarray()

y = df['category']

In [ ]:
X

In [ ]:
X.shape

In [ ]:
#set the tracking url
mlflow.set_tracking_uri('http://ec2-3-88-103-118.compute-1.amazonaws.com:5000/')


#set or create the experiment
mlflow.set_experiment('RF_baseline')

In [ ]:
##install boto3 to communicate the notebook with aws server
!pip install boto3 -q

In [ ]:
!pip install awscli -q

In [ ]:
#setup aws configuration
!aws configure

In [ ]:
##data split and logging
X_train,X_test,y_train,y_test = train_test_split(X,
                                                 y,
                                                 test_size=0.2,
                                                 random_state=42,
                                                 stratify=y)

#train baseline model
with mlflow.start_run() as run:
    #logging description for the run
    mlflow.set_tag('mlflow.runName','RandomeForest_Baseline_Traintestsplit')
    mlflow.set_tag('experiment_type','baseline')
    mlflow.set_tag('model_type','RandomForestClassifier')

    #add a description
    mlflow.set_tag('description','Baseline RF model for sentiment analysis using BOW with ')

    #log parameters
    n_estimators = 200
    max_depth = 15

    mlflow.log_param('n_estimators',n_estimators)
    mlflow.log_param('max_depth',max_depth)

    #initialize the model
    model = RandomForestClassifier(n_estimators=n_estimators,
                                   max_depth=max_depth,
                                   random_state=42)
    #fit
    model.fit(X_train,y_train)

    #make predictions
    y_pred = model.predict(X_test)

    #log metrics
    accuracy = accuracy_score(y_test,y_pred)
    mlflow.log_metric('accuracy',accuracy)

    #classification report & precision_recall_support
    report_dict = classification_report(y_test,y_pred,output_dict=True)

    for label,metrics in report_dict.items():
        if isinstance(metrics,dict):
            for metric,value in metrics.items():
                mlflow.log_metric(f'{label}_{metric}',value)
    #confusion matrix
    conf_matrix = confusion_matrix(y_test,y_pred)
    plt.figure(figsize=(10,8))
    sns.heatmap(conf_matrix,annot=True,fmt='d',cmap='magma')

    #save and log the confusion matrix
    plt.savefig('confusion_matrix.png')
    mlflow.log_artifact('/content/confusion_matrix.png')

    #log the RF model
    mlflow.sklearn.log_model(sk_model=model,
                             name='random_forest_model')
    #logging dataset
    df.to_csv('dataset.csv',index=False)
    mlflow.log_artifact('/content/dataset.csv')
#display final accuracy
print(f'Accuracy:',{accuracy})

In [ ]:
print(classification_report(y_test,y_pred))

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer

# Ensure NLTK data is downloaded if not already
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
try:
    WordNetLemmatizer()
except LookupError:
    nltk.download('wordnet')

def preprocess_and_save_dataframe(input_df: pd.DataFrame, output_csv_path: str) -> pd.DataFrame:
    """
    Applies a series of text preprocessing steps to the 'clean_comment' column of a DataFrame,
    removes missing values and duplicates, and saves the processed DataFrame to a CSV file.

    Args:
        input_df (pd.DataFrame): The original DataFrame to be processed.
        output_csv_path (str): The file path where the processed DataFrame will be saved.

    Returns:
        pd.DataFrame: The processed DataFrame.
    """
    processed_df = input_df.copy()

    # 1. Handle missing values
    processed_df.dropna(inplace=True)

    # 2. Handle duplicates
    processed_df.drop_duplicates(inplace=True)

    # 3. Handle empty or whitespace-only comments
    processed_df = processed_df[~(processed_df['clean_comment'].str.strip() == '')]

    # Ensure 'clean_comment' is string type before applying string methods
    processed_df['clean_comment'] = processed_df['clean_comment'].astype(str)

    # 4. Convert to lowercase
    processed_df['clean_comment'] = processed_df['clean_comment'].str.lower()

    # 5. Strip leading/trailing spaces (redundant after step 3, but harmless)
    processed_df['clean_comment'] = processed_df['clean_comment'].str.strip()

    # 6. Remove newlines
    processed_df['clean_comment'] = processed_df['clean_comment'].str.replace('\n', '', regex=False)

    # 7. Remove special characters (keeping alphanumeric, spaces, and specified punctuation)
    processed_df['clean_comment'] = processed_df['clean_comment'].apply(lambda x: re.sub(r'[^A-Za-z0-9\s!?,.]', '', str(x)))

    # 8. Remove stopwords (with exceptions as defined in previous steps)
    custom_stopwords = set(stopwords.words('english')) - {'not', 'no', 'however', 'but', 'yet'}
    processed_df['clean_comment'] = processed_df['clean_comment'].apply(
        lambda x: ' '.join([word for word in x.split() if word not in custom_stopwords])
    )

    # 9. Lemmatization
    lemmatizer = WordNetLemmatizer()
    processed_df['clean_comment'] = processed_df['clean_comment'].apply(
        lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()])
    )

    # Save to CSV
    processed_df.to_csv(output_csv_path, index=False)
    print(f"Processed DataFrame saved to {output_csv_path}")

    return processed_df

# Example usage of the function
# Assuming 'df' is the original DataFrame after initial load from 'Reddit_Data.csv'
# To demonstrate, let's reload the initial data to ensure the function works from a fresh state,
# but typically you would pass the 'df' that's currently in your session.
# For this example, I'll pass the current 'df' as it is in the notebook state.

processed_df_output = preprocess_and_save_dataframe(df.copy(), 'processed_reddit_data.csv')
print("Processed DataFrame head:")
processed_df_output.head()

In [ ]:
import mlflow
from sklearn.feature_extraction.text import TfidfTransformer, CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import tempfile
import os

def run_mlflow_experiment_with_vectorizer(
    processed_df: pd.DataFrame,
    vectorizer_type: str = 'CountVectorizer',
    ngram_range: tuple = (1, 1),
    max_features: int = 10000,
    vectorizer_name: str = 'default_vectorizer',
    n_estimators: int = 200,
    max_depth: int = 15
):
    """
    Runs an MLflow experiment for sentiment analysis with configurable vectorization and model parameters.

    Args:
        processed_df (pd.DataFrame): The preprocessed DataFrame containing 'clean_comment' and 'category' columns.
        vectorizer_type (str): Type of vectorizer to use ('CountVectorizer' or 'TfidfVectorizer').
        ngram_range (tuple): The n-gram range for the vectorizer (e.g., (1, 1) for unigrams).
        max_features (int): The maximum number of features for the vectorizer.
        vectorizer_name (str): A descriptive name for the vectorizer configuration.
        n_estimators (int): Number of trees in the Random Forest.
        max_depth (int): Maximum depth of the trees in the Random Forest.
    """

    # 1. Initialize Vectorizer
    if vectorizer_type == 'CountVectorizer':
        vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=max_features)
    elif vectorizer_type == 'TfidfVectorizer':
        vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    else:
        raise ValueError("vectorizer_type must be 'CountVectorizer' or 'TfidfVectorizer'")

    # 2. Vectorization
    X = vectorizer.fit_transform(processed_df['clean_comment']).toarray()
    y = processed_df['category']

    # 3. Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 4. MLflow Run
    with mlflow.start_run() as run:
        # Logging description for the run
        mlflow.set_tag('mlflow.runName', f'RF_Sentiment_{vectorizer_name}_{vectorizer_type}_{ngram_range}')
        mlflow.set_tag('experiment_type', 'vectorizer_comparison_FE')
        mlflow.set_tag('model_type', 'RandomForestClassifier')
        mlflow.set_tag('description', f'RF model with {vectorizer_type} using {vectorizer_name} vectorizer configuration.')

        # Log vectorizer parameters
        mlflow.log_param('vectorizer_type', vectorizer_type)
        mlflow.log_param('ngram_range', str(ngram_range)) # Convert tuple to string for logging
        mlflow.log_param('max_features', max_features)
        mlflow.log_param('vectorizer_name', vectorizer_name)

        # Log model parameters
        mlflow.log_param('n_estimators', n_estimators)
        mlflow.log_param('max_depth', max_depth)

        # Initialize and train the model
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42
        )
        model.fit(X_train, y_train)

        # Make predictions
        y_pred = model.predict(X_test)

        # Log metrics
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric('accuracy', accuracy)

        report_dict = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in report_dict.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f'{label}_{metric}', value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(10, 8))
        sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='magma')
        # Save to a temporary file for logging
        with tempfile.TemporaryDirectory() as tmpdir:
            cm_path = os.path.join(tmpdir, 'confusion_matrix.png')
            plt.savefig(cm_path)
            mlflow.log_artifact(cm_path)
        plt.close()

        # logging the RF model
        mlflow.sklearn.log_model(sk_model=model,
                                 artifact_path='random_forest_model')

        # Log the processed dataset as an artifact
        with tempfile.TemporaryDirectory() as tmpdir:
            dataset_path = os.path.join(tmpdir, 'processed_dataset.csv')
            processed_df.to_csv(dataset_path, index=False)
            mlflow.log_artifact(dataset_path)

        print(f"MLflow Run finished. Run ID: {run.info.run_id}, Accuracy: {accuracy}")


In [ ]:
#lets run the exp with different ngram ranges
ngram_ranges = [(1,1),(1,2),(1,3)]

In [ ]:
for ngram_range in ngram_ranges:
    # Run with CountVectorizer (default parameters)
    run_mlflow_experiment_with_vectorizer(processed_df_output,
                                      vectorizer_type='CountVectorizer',
                                      ngram_range=ngram_range,
                                      max_features=10000,
                                      vectorizer_name='unigram_10k')
    # Run with TfidfVectorizer and bigrams
    run_mlflow_experiment_with_vectorizer(processed_df_output,
                                      vectorizer_type='TfidfVectorizer',
                                      ngram_range=ngram_range,
                                      max_features=10000,
                                      vectorizer_name='tfidf_bigram_5k',
                                      n_estimators=150,
                                      max_depth=20)

In [ ]:
##from the result on the mlflow server
##the tfidfvectorizer is better at ngram_range 1,3
#so i chose that one and now i will check the max_features

#just select the function where the parameter is tfidfvectorizer
# Run with TfidfVectorizer and set ngram

max_features = [1000,2000,3000,4000,5000,6000,7000,8000,9000,10000]

for max_f in max_features:
    run_mlflow_experiment_with_vectorizer(processed_df_output,
                                      vectorizer_type='TfidfVectorizer',
                                      ngram_range=(1,3), #set to (1,3)
                                      max_features=max_f,
                                      vectorizer_name=f'max_feature_{max_f}', #do this to register different maxf values
                                      n_estimators=150,
                                      max_depth=20)

In [ ]:
##set new experiment name for handling data imbalance
mlflow.set_experiment('rf-handling-imbalanced-dataset')

In [ ]:
##handling the imbalanced dataset with imblearn over and under sampling techniques
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE,ADASYN
from imblearn.combine import SMOTEENN

In [ ]:
#this fnction takes the slected vectorizer and runs different sampling
#techniques
def run_mlflow_experiment_with_imbalance_handling(
    processed_df: pd.DataFrame,
    imbalance_method: str = 'none',
    n_estimators: int = 150,
    max_depth: int = 20,
    vectorizer_max_features: int = 1000,
    vectorizer_ngram_range: tuple = (1, 3)
):
    """
    Runs an MLflow experiment for sentiment analysis with configurable imbalance handling methods.
    Uses TfidfVectorizer with specified optimal parameters.

    Args:
        processed_df (pd.DataFrame): The preprocessed DataFrame containing 'clean_comment' and 'category'.
        imbalance_method (str): The method for handling imbalanced data ('none', 'RandomUnderSampler', 'SMOTE', 'ADASYN', 'SMOTEENN').
        n_estimators (int): Number of trees in the Random Forest.
        max_depth (int): Maximum depth of the trees in the Random Forest.
        vectorizer_max_features (int): The maximum number of features for the TfidfVectorizer.
        vectorizer_ngram_range (tuple): The n-gram range for the TfidfVectorizer.
    """

    # 1. Initialize TfidfVectorizer with preferred parameters
    vectorizer = TfidfVectorizer(
        ngram_range=vectorizer_ngram_range,
        max_features=vectorizer_max_features
    )

    # 2. Vectorization
    X = vectorizer.fit_transform(processed_df['clean_comment'])
    y = processed_df['category']

    # 3. Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 4. Apply Imbalance Handling Method
    sampler = None
    if imbalance_method == 'RandomUnderSampler':
        sampler = RandomUnderSampler(random_state=42)
    elif imbalance_method == 'SMOTE':
        sampler = SMOTE(random_state=42)
    elif imbalance_method == 'ADASYN':
        sampler = ADASYN(random_state=42)
    elif imbalance_method == 'SMOTEENN':
        sampler = SMOTEENN(random_state=42)
    elif imbalance_method != 'none':
        raise ValueError("Invalid imbalance_method. Choose from 'none', 'RandomUnderSampler', 'SMOTE', 'ADASYN', 'SMOTEENN'")

    if sampler:
        print(f"Applying {imbalance_method}...")
        X_train_resampled, y_train_resampled = sampler.fit_resample(X_train, y_train)
    else:
        #if no sampling technique is chosen
        X_train_resampled, y_train_resampled = X_train, y_train

    # 5. MLflow Run
    with mlflow.start_run() as run:
        # Logging description for the run
        mlflow.set_tag('mlflow.runName', f'RF_Imbalance_{imbalance_method}')
        mlflow.set_tag('experiment_type', 'imbalance_handling_comparison')
        mlflow.set_tag('model_type', 'RandomForestClassifier')
        mlflow.set_tag('description', f'RF model with TfidfVectorizer and {imbalance_method} sampling.')

        # Log vectorizer parameters
        mlflow.log_param('vectorizer_type', 'TfidfVectorizer')
        mlflow.log_param('ngram_range', str(vectorizer_ngram_range))
        mlflow.log_param('max_features', vectorizer_max_features)


        # Log imbalance handling parameter
        mlflow.log_param('imbalance_method', imbalance_method)

        # Log model parameters
        mlflow.log_param('n_estimators', n_estimators)
        mlflow.log_param('max_depth', max_depth)

        # Initialize and train the model
        model = RandomForestClassifier(
            n_estimators=n_estimators, max_depth=max_depth, random_state=42
        )
        model.fit(X_train_resampled, y_train_resampled)

        # Make predictions
        y_pred = model.predict(X_test)

        # Log metrics
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric('accuracy', accuracy)

        report_dict = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in report_dict.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f'{label}_{metric}', value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(10, 8))
        sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='magma')
        with tempfile.TemporaryDirectory() as tmpdir:
            cm_path = os.path.join(tmpdir, f'confusion_matrix_{imbalance_method}.png')
            plt.savefig(cm_path)
            mlflow.log_artifact(cm_path)
        plt.close()

        # Log the RF model
        mlflow.sklearn.log_model(sk_model=model, artifact_path='random_forest_model')

        # Log the processed dataset as an artifact
        with tempfile.TemporaryDirectory() as tmpdir:
            dataset_path = os.path.join(tmpdir, 'processed_dataset.csv')
            processed_df.to_csv(dataset_path, index=False)
            mlflow.log_artifact(dataset_path)

        print(f"MLflow Run finished for {imbalance_method}. Run ID: {run.info.run_id}, Accuracy: {accuracy}")

In [ ]:
# Run with different sampling methods

sampling_methods = ['none', 'RandomUnderSampler', 'SMOTE', 'ADASYN', 'SMOTEENN']

#run the experiment for all sampling methods
for sample_method in sampling_methods:
    run_mlflow_experiment_with_imbalance_handling(processed_df_output,
                                                  imbalance_method=f'{sample_method}')


In [ ]:
##using a different model called - stackingclassifier
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import classification_report,accuracy_score,confusion_matrix
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
# Set a new MLflow experiment for stacking classifiers
mlflow.set_experiment('rf-stacking-classifier')

In [ ]:
#using the tfidfvectorizer, we setup the new ml models
def run_mlflow_stacking_experiment(
    processed_df: pd.DataFrame,
    n_estimators_rf: int = 150,
    max_depth_rf: int = 20,
    vectorizer_max_features: int = 1000,
    vectorizer_ngram_range: tuple = (1, 3)
):
    """
    Runs an MLflow experiment using a StackingClassifier with specified base estimators.
    Applies RandomUnderSampler and TfidfVectorizer with optimal parameters.

    Args:
        processed_df (pd.DataFrame): The preprocessed DataFrame containing 'clean_comment' and 'category'.
        n_estimators_rf (int): Number of trees for the Random Forest meta-estimator.
        max_depth_rf (int): Maximum depth for the Random Forest meta-estimator.
        vectorizer_max_features (int): The maximum number of features for the TfidfVectorizer.
        vectorizer_ngram_range (tuple): The n-gram range for the TfidfVectorizer.
    """

    # 1. Initialize TfidfVectorizer with preferred parameters
    vectorizer = TfidfVectorizer(
        ngram_range=vectorizer_ngram_range,
        max_features=vectorizer_max_features
    )

    # 2. Vectorization
    X = vectorizer.fit_transform(processed_df['clean_comment'])
    y = processed_df['category']

    # 3. Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 4. Apply RandomUnderSampler
    print("Applying RandomUnderSampler...")
    rus = RandomUnderSampler(random_state=42)
    X_train_resampled, y_train_resampled = rus.fit_resample(X_train, y_train)

    # 5. defining base estimators
    estimators = [
        ('lgbm', LGBMClassifier(random_state=42,
                                n_estimators=367,
                                learning_rate=0.1,
                                num_classes=len(df['category']),
                                metric='multi_logloss',
                                is_unbalanced=True,
                                class_weight='balanced',
                                objective='multiclass',
                                reg_alpha=0.1,
                                ref_lambda=0.1,
                                max_depth=20,
                                )),
        ('knn', KNeighborsClassifier(n_neighbors=5)),
        ('lr', LogisticRegression(random_state=42,
                                  solver='liblinear',
                                  C=1.0,max_iter=1000,
                                  class_weight='balanced',))
    ]

    # 6. Initialize StackingClassifier
    # Use RandomForestClassifier as the final estimator
    st_clf = StackingClassifier(
        estimators=estimators,
        final_estimator=RandomForestClassifier(
            n_estimators=n_estimators_rf, max_depth=max_depth_rf, random_state=42
        ),
        cv=5,
        n_jobs=-1 # setting to -1 means it uses all available cores
    )

    # 7. MLflow Run
    with mlflow.start_run() as run:
        # Logging description for the run
        mlflow.set_tag('mlflow.runName', 'StackingClassifier_RUS_OptimalTFIDF')
        mlflow.set_tag('experiment_type', 'model_comparison_stacking')
        mlflow.set_tag('model_type', 'StackingClassifier')
        mlflow.set_tag('description', 'StackingClassifier with LGBM, KNN, LR base estimators and RF final, with RUS and optimal TfidfVectorizer.')

        # Log vectorizer parameters
        mlflow.log_param('vectorizer_type', 'TfidfVectorizer')
        mlflow.log_param('ngram_range', str(vectorizer_ngram_range))
        mlflow.log_param('max_features', vectorizer_max_features)

        # Log imbalance handling parameter
        mlflow.log_param('imbalance_method', 'RandomUnderSampler')

        # Log final estimator parameters
        mlflow.log_param('final_estimator_n_estimators', n_estimators_rf)
        mlflow.log_param('final_estimator_max_depth', max_depth_rf)

        # Log base estimator types
        mlflow.log_param('base_estimators', '[LGBMClassifier, KNeighborsClassifier, LogisticRegression]')

        # Train the StackingClassifier
        print("Training StackingClassifier...")
        st_clf.fit(X_train_resampled, y_train_resampled)

        # Make predictions
        y_pred = st_clf.predict(X_test)

        # Log metrics
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric('accuracy', accuracy)

        report_dict = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in report_dict.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f'{label}_{metric}', value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(10, 8))
        sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='magma')
        with tempfile.TemporaryDirectory() as tmpdir:
            cm_path = os.path.join(tmpdir, 'confusion_matrix_stacking.png')
            plt.savefig(cm_path)
            mlflow.log_artifact(cm_path)
        plt.close()

        # Log the StackingClassifier model
        mlflow.sklearn.log_model(sk_model=st_clf, artifact_path='stacking_classifier_model')

        # Log the processed dataset as an artifact
        with tempfile.TemporaryDirectory() as tmpdir:
            dataset_path = os.path.join(tmpdir, 'processed_dataset.csv')
            processed_df.to_csv(dataset_path, index=False)
            mlflow.log_artifact(dataset_path)

        print(f"MLflow Run finished for StackingClassifier. Run ID: {run.info.run_id}, Accuracy: {accuracy}")

In [ ]:
# Run the stacking experiment
run_mlflow_stacking_experiment(processed_df_output)